In [1]:
# --- Colab bootstrap -------------------------------------------------------
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    !rm -rf /tmp/cbet6e
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    sys.path.insert(0, '.')
# ---------------------------------------------------------------------------


# The octanol-water partition coefficient, and what it is for (Illustrations 11.4-2 and 11.4-3)

Two illustrations that use the same number for opposite purposes.

**Illustration 11.4-2** estimates $K_{\rm OW}$ for benzo[a]pyrene -- a combustion
product and a carcinogen -- from nothing but its activity coefficient in water. That is
the environmental-engineering use: a chemical's partitioning between water and organic
matter decides where it ends up in a river, a soil, or a fish.

**Illustration 11.4-3** goes the other way. Given $K_{\rm OW} = 65.5$ for
benzylpenicillin, how much antibiotic ends up in each phase of a 25 mL + 25 mL
extraction? That is the process use, and the answer is a one-line mass balance.

**Why octanol.** Nothing about $n$-octanol is fundamental. It is a stand-in for
"organic matter" -- a molecule with a hydroxyl group and a long tail, so it dissolves a
useful amount of water (26 mol %) while being essentially insoluble in it. The
convention is old, the measurements are abundant, and correlations against it are how
environmental partitioning is estimated in practice.

**And $K_{\rm OW}$ is an activity coefficient ratio in disguise**, which is the reason
it belongs in this chapter. Equation 11.4-11 is

$$K_{{\rm OW},i} = \frac{C^{\rm O}}{C^{\rm W}}\,
   \frac{\gamma_i^{W,\infty}}{\gamma_i^{O,\infty}}$$

and everything the section does with it follows from putting numbers on those three
factors.

SIS is Stanley I. Sandler, *Chemical, Biochemical, and Engineering Thermodynamics*.

Eric Furst
August 2026

In [2]:

import sys; sys.path.insert(0, "..")
import numpy as np

from thermo.partition import (kow_from_gamma, gamma_from_kow, solute_split,
                              OCTANOL_WATER)

R = 8.314
T = 298.15

# --- Illustration 11.4-2 ----------------------------------------------------
GAMMA_BP = 3.76e8              # benzo[a]pyrene in water, from Illustration 12.1-3
LOG_KOW_REPORTED = 6.04        # the measured value, K_OW = 1.1e6

print("  Where Eq. 11.4-12's 0.0228 comes from, and it is not fitted")
CO_CW = ((OCTANOL_WATER["rho_octanol"] / OCTANOL_WATER["mw_octanol"])
         / (OCTANOL_WATER["rho_water"] / OCTANOL_WATER["mw_water"]))
print(f"    C^O/C^W = (0.827/130.22)/(1/18) = {CO_CW:.4f}        SIS 0.114")
print(f"    divided by gamma^(O,inf) = 5:     {CO_CW/5:.5f}      SIS 0.0228")
print(f"    log10 of that:                    {np.log10(CO_CW/5):.3f}       SIS -1.642")

print("\n  Illustration 11.4-2, both correlations")
for eq in ("11.4-12", "11.4-13"):
    K = float(kow_from_gamma(GAMMA_BP, eq))
    print(f"    Eq. {eq}: K_OW = {K:.3g}, log10 K_OW = {np.log10(K):.2f}")
print(f"    SIS:       8.57e6 and log 6.93;  2.69e6 and log 6.43")
print(f"    reported:  log10 K_OW = {LOG_KOW_REPORTED}, K_OW = 1.1e6")

K13 = float(kow_from_gamma(GAMMA_BP, "11.4-13"))
print(f"\n    Eq. 11.4-13 is high by a factor of {K13/1.1e6:.1f}"
      f"      SIS 'within a factor of 2.5'")
print(f"    Eq. 11.4-12 is high by a factor of"
      f" {float(kow_from_gamma(GAMMA_BP, '11.4-12'))/1.1e6:.0f}")
print(f"\n  Inverted, the measured K_OW implies gamma^(W,inf) ="
      f" {float(gamma_from_kow(1.1e6)):.2e}")
print(f"  against the {GAMMA_BP:.2e} the calculation started from"
      f" -- a factor of {GAMMA_BP/float(gamma_from_kow(1.1e6)):.1f}.")

  Where Eq. 11.4-12's 0.0228 comes from, and it is not fitted
    C^O/C^W = (0.827/130.22)/(1/18) = 0.1143        SIS 0.114
    divided by gamma^(O,inf) = 5:     0.02286      SIS 0.0228
    log10 of that:                    -1.641       SIS -1.642

  Illustration 11.4-2, both correlations
    Eq. 11.4-12: K_OW = 8.57e+06, log10 K_OW = 6.93
    Eq. 11.4-13: K_OW = 2.66e+06, log10 K_OW = 6.43
    SIS:       8.57e6 and log 6.93;  2.69e6 and log 6.43
    reported:  log10 K_OW = 6.04, K_OW = 1.1e6

    Eq. 11.4-13 is high by a factor of 2.4      SIS 'within a factor of 2.5'
    Eq. 11.4-12 is high by a factor of 8

  Inverted, the measured K_OW implies gamma^(W,inf) = 1.25e+08
  against the 3.76e+08 the calculation started from -- a factor of 3.0.



## Where $3.76\times10^8$ comes from, and whether it checks out

Illustration 11.4-2 takes the infinite-dilution activity coefficient of benzo[a]pyrene
in water as given, from an illustration two chapters later. That illustration gets it
from a solubility, and the route matters here because the same route is what the next
notebook uses on 32 substances at once.

Benzo[a]pyrene is a **solid** at 25 °C, so its activity coefficient in a saturated
aqueous solution is not $1/x^{\rm sat}$. The equilibrium condition is

$$x^{\rm sat}\gamma^{\infty} = \frac{f^{\rm S}(T,P)}{f^{\rm L}(T,P)}
  = \exp\left[-\frac{\Delta_{\rm fus}H}{R}\left(\frac{1}{T}-\frac{1}{T_m}\right)\right]$$

which is Eq. 9.7-8a, and the right side is smaller than one. Taking $\gamma = 1/x^{\rm sat}$
instead would overestimate the activity coefficient by exactly that factor.

Illustration 12.1-3 gives the three numbers: melting point 178.1 °C, heat of fusion
15.1 kJ/mol, aqueous solubility $x_{\rm BP} = 3.37\times10^{-10}$.

In [3]:

TM_BP = 178.1 + 273.15         # K
DH_FUS_BP = 15100.0            # J/mol
X_SAT_BP = 3.37e-10            # mole fraction, Illustration 12.1-3

fugacity_ratio = np.exp(-DH_FUS_BP / R * (1 / T - 1 / TM_BP))
gamma_check = fugacity_ratio / X_SAT_BP

print("  Illustration 12.1-3, recomputed here because Illustration 11.4-2 rests on it")
print(f"    f_solid/f_liquid  = {fugacity_ratio:.4f}")
print(f"    gamma^(W,inf)     = {gamma_check:.3e}      SIS 3.76e8")
print(f"    1/x_sat would be  = {1/X_SAT_BP:.3e}"
      f"      high by a factor of {1/fugacity_ratio:.1f}")

print("\n  And that factor is the entire reason a melting point appears in a table of")
print("  partition coefficients: for a solid solute, a solubility does not give an")
print(f"  activity coefficient until the fugacity ratio is divided out. Ignoring it")
print(f"  here would raise log10 K_OW by"
      f" {0.806*np.log10(1/fugacity_ratio):.2f} through Eq. 11.4-13.")

  Illustration 12.1-3, recomputed here because Illustration 11.4-2 rests on it
    f_solid/f_liquid  = 0.1266
    gamma^(W,inf)     = 3.757e+08      SIS 3.76e8
    1/x_sat would be  = 2.967e+09      high by a factor of 7.9

  And that factor is the entire reason a melting point appears in a table of
  partition coefficients: for a solid solute, a solubility does not give an
  activity coefficient until the fugacity ratio is divided out. Ignoring it
  here would raise log10 K_OW by 0.72 through Eq. 11.4-13.


The chain closes: $0.1266 / 3.37\times10^{-10} = 3.757\times10^{8}$, which is the
$3.76\times10^8$ Illustration 11.4-2 quotes to three figures.

**One cross-reference to fix.** Illustration 11.4-2 opens *"In Illustration 12.3 it
will be shown that the value of the infinite-dilution activity coefficient of
benzo[a]pyrene in water is $3.76\times10^8$."* The illustration is **12.1-3**, not 12.3.
The printed page has it right; the Word source says 12.3, so the fix was
made in typesetting and never came back to the file.

## Illustration 11.4-3: using $K_{\rm OW}$ to purify something

Benzylpenicillin, 200 mg of it, shaken with 25 mL of $n$-octanol and 25 mL of water.
The octanol-rich phase takes up water -- 26 mol %, the same figure Illustration 11.2-4
gives -- so the two phase volumes are not the two volumes charged, and the volumes are
what the mass balance needs.

In [4]:

# --- Illustration 11.4-3 ----------------------------------------------------
MW_OCT, RHO_OCT = 130.23, 0.826          # g/mol, g/cc, as this illustration prints them
MW_WATER_ILL = 18.0
KOW_PENICILLIN = 65.5
MASS_CHARGED = 0.200                      # g
V_OCT_CHARGED = 25.0                      # mL
V_WATER_CHARGED = 25.0                    # mL

n_octanol = V_OCT_CHARGED * RHO_OCT / MW_OCT
n_water_in_oct = n_octanol * 0.26 / 0.74
V_O = n_octanol * MW_OCT / RHO_OCT + n_water_in_oct * MW_WATER_ILL / 1.0
V_W = V_WATER_CHARGED - n_water_in_oct * MW_WATER_ILL / 1.0

print("  The two phase volumes")
print(f"    n_octanol            = {n_octanol:.4f} mol      SIS 0.1586")
print(f"    water in that phase  = {n_water_in_oct:.4f} mol      SIS 0.0557")
print(f"    V^O = {V_O:.4f} mL                          SIS 26.0028")
print(f"    V^W = {V_W:.4f} mL                          SIS 23.997")

print("\n  The mass balance, at both molecular weights the illustration prints")
for MW_P, where in ((334.5, "the Data line"), (334.4, "the solution")):
    n_total = MASS_CHARGED / MW_P
    C_O, C_W = solute_split(n_total, V_O, V_W, KOW_PENICILLIN)
    print(f"    MW {MW_P} ({where}): n = {n_total:.4e} mol,"
          f"  C^W = {C_W*1e3*MW_P:.4f} mg/mL,  C^O = {C_O*1e3*MW_P:.4f} mg/mL")
print(f"    SIS: n = 5.981e-4 mol, C^W = 0.1158 mg/mL, C^O = 7.585 mg/mL")

print(f"\n  denominator V^W + K V^O = {V_W + KOW_PENICILLIN*V_O:.2f} mL"
      f"        SIS 1727.52")
print(f"  and the printed value needs V^O = "
      f"{(1727.52 - 23.997)/KOW_PENICILLIN:.4f} mL rather than {V_O:.4f}")

n_total = MASS_CHARGED / 334.4
C_O, C_W = solute_split(n_total, V_O, V_W, KOW_PENICILLIN)
recovered = C_O * V_O / n_total
print(f"\n  Fraction of the antibiotic recovered in the octanol phase:"
      f" {100*recovered:.1f} %")
print(f"  left behind in the water:                            "
      f" {100*(1-recovered):.1f} %")

  The two phase volumes
    n_octanol            = 0.1586 mol      SIS 0.1586
    water in that phase  = 0.0557 mol      SIS 0.0557
    V^O = 26.0028 mL                          SIS 26.0028
    V^W = 23.9972 mL                          SIS 23.997

  The mass balance, at both molecular weights the illustration prints
    MW 334.5 (the Data line): n = 5.9791e-04 mol,  C^W = 0.1158 mg/mL,  C^O = 7.5846 mg/mL
    MW 334.4 (the solution): n = 5.9809e-04 mol,  C^W = 0.1158 mg/mL,  C^O = 7.5846 mg/mL
    SIS: n = 5.981e-4 mol, C^W = 0.1158 mg/mL, C^O = 7.585 mg/mL

  denominator V^W + K V^O = 1727.18 mL        SIS 1727.52
  and the printed value needs V^O = 26.0080 mL rather than 26.0028

  Fraction of the antibiotic recovered in the octanol phase: 98.6 %
  left behind in the water:                             1.4 %



**Two printed slips, neither of which changes an answer.** The Data line gives the
molecular weight of benzylpenicillin as 334.5 and the solution divides by 334.4 (the
formula weight is 334.39, so the solution is right and the Data line needs correcting).
And the mass balance carries $65.5 \times 26.008$ where the volume calculation three
lines above gives 26.0028 -- a transposition, which is why the printed denominator is
1727.52 where it should be 1727.18. Both wash out in the third significant figure and the
printed concentrations, 0.1158 and 7.585 mg/mL, are what this notebook gets.

**The number to take away is 98.6 %.** A partition coefficient of 65 does *not*
mean 65 parts in 66: it means 65 times the *concentration*, and with nearly equal phase
volumes that is 98.6 % recovery in one contact. Getting the last 1.4 % is what
multi-stage extraction is for, and Illustration 11.2-9 is that calculation.

## Your turn

1. Repeat Illustration 11.4-3 with 5 mL of octanol instead of 25 mL. How much
   benzylpenicillin is recovered, and how much more concentrated is the product? That
   trade -- recovery against concentration -- is the whole design problem of extraction.
2. Then do it in two stages of 12.5 mL each and compare with one stage of 25 mL, the way
   Illustration 11.2-9 compares two 1 kg washes with one 3 kg wash.
3. Eq. 11.4-12 takes $\gamma_i^{O,\infty} \approx 5$ for every solute. Invert
   Eq. 11.4-11 with the measured $K_{\rm OW}$ of benzo[a]pyrene and its
   $\gamma^{W,\infty} = 3.76\times10^8$ to get the actual value. Is 5 a good guess?
4. Illustration 11.4-3 assumes the benzylpenicillin does not change the mutual
   solubility of octanol and water. At the concentrations computed above, how many moles
   of antibiotic are in each phase compared with the moles of octanol and water? Is the
   assumption safe?
5. Benzylpenicillin is an acid ($pK_a \approx 2.8$) and partitions as the neutral
   molecule. Sketch what $K_{\rm OW}$ measured at pH 7 would look like compared with the
   65.5 used here, and say which one belongs in a purification calculation.